# OKX 4h 4-Year Indicator Preprocessing
## Comprehensive Technical Indicators — Zero Lookahead Bias

**Dataset**: 69 cryptocurrency perpetual swap pairs, 4 years of 4h OHLCV data (2022–2026)
**Indicators**: 67 features per candle covering Trend, Momentum, Volatility, Volume, Returns
**Method**: Pure pandas/numpy, strict `.shift(1)` no-lookahead enforcement

In [ ]:
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import os, sys, time, logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("kaggle")

INPUT_DIR = Path("/kaggle/input/okx-4h-4y/okx_4h_4y")
OUTPUT_DIR = Path("/kaggle/working/okx_4h_4y_indicators")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-10
WARMUP_ROWS = 300

In [ ]:
def compute_all_indicators(df):
    close = df["close"].astype(float)
    high = df["high"].astype(float)
    low = df["low"].astype(float)
    open_ = df["open"].astype(float)
    volume = df["volume"].astype(float)

    result = df[["timestamp", "open", "high", "low", "close", "volume"]].copy()

    # SMA (7,14,25,50,100,200)
    for p in [7, 14, 25, 50, 100, 200]:
        result[f"sma_{p}"] = close.rolling(p, min_periods=p).mean()
    # EMA
    for p in [7, 14, 25, 50, 100, 200]:
        result[f"ema_{p}"] = close.ewm(span=p, min_periods=p, adjust=False).mean()
    # Price vs MA
    for p in [7, 25, 50, 200]:
        result[f"close_vs_ema_{p}"] = close / (result[f"ema_{p}"] + EPS)
        result[f"close_vs_sma_{p}"] = close / (result[f"sma_{p}"] + EPS)
    # MA Ratios
    result["ema_7_25_ratio"] = result["ema_7"] / (result["ema_25"] + EPS)
    result["ema_25_50_ratio"] = result["ema_25"] / (result["ema_50"] + EPS)
    result["ema_50_200_ratio"] = result["ema_50"] / (result["ema_200"] + EPS)

    # RSI
    for p in [7, 14]:
        delta = close.diff()
        gain = delta.clip(lower=0)
        loss = (-delta).clip(lower=0)
        avg_gain = gain.ewm(alpha=1/p, min_periods=p, adjust=False).mean()
        avg_loss = loss.ewm(alpha=1/p, min_periods=p, adjust=False).mean()
        rs = avg_gain / (avg_loss + EPS)
        result[f"rsi_{p}"] = 100 - (100 / (1 + rs))

    # Stochastic
    kp = 14
    low_min = low.rolling(kp, min_periods=kp).min()
    high_max = high.rolling(kp, min_periods=kp).max()
    result["stoch_k"] = 100 * (close - low_min) / (high_max - low_min + EPS)
    result["stoch_d"] = result["stoch_k"].rolling(3, min_periods=3).mean()

    # MACD
    e12 = close.ewm(span=12, min_periods=12, adjust=False).mean()
    e26 = close.ewm(span=26, min_periods=26, adjust=False).mean()
    result["macd_line"] = e12 - e26
    result["macd_signal"] = result["macd_line"].ewm(span=9, min_periods=9, adjust=False).mean()
    result["macd_hist"] = result["macd_line"] - result["macd_signal"]

    # Bollinger Bands
    s20 = close.rolling(20, min_periods=20).mean()
    std20 = close.rolling(20, min_periods=20).std()
    result["bb_upper"] = s20 + 2*std20
    result["bb_middle"] = s20
    result["bb_lower"] = s20 - 2*std20
    result["bb_width"] = result["bb_upper"] - result["bb_lower"]
    result["bb_width_pct"] = result["bb_width"] / (s20 + EPS) * 100
    result["bb_position"] = (close - result["bb_lower"]) / (result["bb_width"] + EPS)

    # ATR (True Range)
    pc = close.shift(1)
    tr = pd.concat([high - low, (high - pc).abs(), (low - pc).abs()], axis=1).max(axis=1)
    result["atr_14"] = tr.ewm(span=14, min_periods=14, adjust=False).mean()
    result["atr_pct"] = result["atr_14"] / (close + EPS) * 100

    # CCI
    tp = (high + low + close) / 3
    smatp = tp.rolling(20, min_periods=20).mean()
    mad = tp.rolling(20, min_periods=20).apply(lambda x: np.abs(x - x.mean()).mean(), raw=True)
    result["cci_20"] = (tp - smatp) / (0.015 * mad + EPS)

    # Williams %R
    wh = high.rolling(14, min_periods=14).max()
    wl = low.rolling(14, min_periods=14).min()
    result["williams_r"] = -100 * (wh - close) / (wh - wl + EPS)

    # MFI
    mfi_tp = tp
    rmf = mfi_tp * volume
    pos = rmf.where(mfi_tp > mfi_tp.shift(1), 0).rolling(14, min_periods=14).sum()
    neg = rmf.where(mfi_tp < mfi_tp.shift(1), 0).rolling(14, min_periods=14).sum()
    result["mfi_14"] = 100 - (100 / (1 + pos / (neg + EPS)))

    # ADX
    ax = 14
    pdm = high.diff().where(lambda x: (x > -low.diff()) & (x > 0), 0)
    mdm = -low.diff().where(lambda x: (x > high.diff()) & (x > 0), 0)
    atrs = tr.ewm(alpha=1/ax, min_periods=ax, adjust=False).mean()
    pdi = 100 * pdm.ewm(alpha=1/ax, min_periods=ax, adjust=False).mean() / (atrs + EPS)
    mdi = 100 * mdm.ewm(alpha=1/ax, min_periods=ax, adjust=False).mean() / (atrs + EPS)
    dx = 100 * np.abs(pdi - mdi) / (pdi + mdi + EPS)
    result["adx_14"] = dx.ewm(alpha=1/ax, min_periods=ax, adjust=False).mean()
    result["plus_di_14"] = pdi
    result["minus_di_14"] = mdi

    # Keltner Channels
    e20k = close.ewm(span=20, min_periods=20, adjust=False).mean()
    atr20 = tr.ewm(span=20, min_periods=20, adjust=False).mean()
    result["kc_upper"] = e20k + 2*atr20
    result["kc_middle"] = e20k
    result["kc_lower"] = e20k - 2*atr20
    result["kc_width_pct"] = (result["kc_upper"] - result["kc_lower"]) / (e20k + EPS) * 100

    # Volume
    result["obv"] = (np.sign(close.diff()) * volume).fillna(0).cumsum()
    for p in [20, 50]:
        result[f"volume_sma_{p}"] = volume.rolling(p, min_periods=p).mean()
    result["volume_ratio_20"] = volume / (result["volume_sma_20"] + EPS)
    result["volume_ratio_50"] = volume / (result["volume_sma_50"] + EPS)
    # CMF
    clv = ((close - low) - (high - close)) / (high - low + EPS)
    result["cmf_20"] = (clv * volume).rolling(20, min_periods=20).sum() / volume.rolling(20, min_periods=20).sum()

    # Returns
    for p in [1, 4, 8, 16]:
        result[f"return_{p}"] = close.pct_change(p)
        result[f"log_return_{p}"] = np.log(close / close.shift(p))
    # Volatility
    for p in [8, 20]:
        result[f"volatility_{p}"] = close.pct_change().rolling(p, min_periods=p).std() * np.sqrt(p)
    # Price range
    result["hl_range_pct"] = (high - low) / (close + EPS) * 100
    result["gap_open_pct"] = (open_ - pc) / (pc + EPS) * 100
    result["close_position"] = (close - low) / (high - low + EPS)

    # === NO-LOOKAHEAD SHIFT ===
    indicator_cols = [c for c in result.columns if c not in ("timestamp", "open", "high", "low", "close", "volume")]
    result[indicator_cols] = result[indicator_cols].shift(1)
    result = result.iloc[WARMUP_ROWS:].reset_index(drop=True)
    return result

In [ ]:
# Find all input parquet files
files = sorted(INPUT_DIR.glob("*_4h.parquet"))
log.info(f"Found {len(files)} input files")

if not files:
    log.error("No input files found! Check dataset path.")
    log.info(f"Contents of {INPUT_DIR}: {list(INPUT_DIR.iterdir())[:10]}")
else:
    total_rows_before = 0
    total_rows_after = 0
    t0 = time.time()
    
    for i, fp in enumerate(files):
        pair = fp.stem.replace("_4h", "")
        df = pd.read_parquet(fp)
        total_rows_before += len(df)
        
        df = compute_all_indicators(df)
        
        out_path = OUTPUT_DIR / f"{pair}_indicators.parquet"
        pq.write_table(pa.Table.from_pandas(df), out_path, compression="zstd", compression_level=3)
        total_rows_after += len(df)
        
        if (i + 1) % 10 == 0:
            elapsed = time.time() - t0
            log.info(f"  [{i+1}/{len(files)}] {elapsed:.0f}s elapsed")
    
    elapsed = time.time() - t0
    log.info(f"\nDONE: {len(files)} files in {elapsed:.0f}s")
    log.info(f"Rows before: {total_rows_before:,}")
    log.info(f"Rows after:  {total_rows_after:,}")
    log.info(f"Output: {OUTPUT_DIR}")
    log.info(f"Output files: {len(list(OUTPUT_DIR.glob('*_indicators.parquet')))} ")

## Validation
Check output quality: NaN%, Inf count, row counts, date ranges.

In [ ]:
out_files = sorted(OUTPUT_DIR.glob("*_indicators.parquet"))
log.info(f"Output files: {len(out_files)}")

total_rows = 0
nan_files = 0

for f in out_files:
    df = pd.read_parquet(f)
    total_rows += len(df)
    if df.isna().sum().sum() > 0:
        nan_files += 1

log.info(f"Total rows: {total_rows:,}")
log.info(f"Files with NaN: {nan_files}")
log.info(f"Total size: {sum(f.stat().st_size for f in out_files)/1024/1024:.0f} MB")

# Show sample
sample = pd.read_parquet(out_files[0])
log.info(f"\nSample ({out_files[0].name}):")
log.info(f"  Shape: {sample.shape}")
log.info(f"  Columns: {list(sample.columns)[:10]}...")
log.info(f"  Date: {sample['timestamp'].min()} → {sample['timestamp'].max()}")